### Теперь займемся подготовкой нашего финального датасета, путем мерджа спаршенного датасета со остальными, полученными через IGDB API

Импортируем необходимые библиотеки:

In [143]:
import pandas as pd

Далее, прочтем все имеющиеся датасеты для дальйнешей их обработки и конечного мерджа

In [144]:
df_parsed = pd.read_excel("../../data/df_parsed.xlsx")

In [145]:
df_games_final = pd.read_csv("../../data/df_games_final.csv")
df_popular_final = pd.read_csv("../../data/df_popular_final.csv")
df_web_final = pd.read_csv("../../data/df_web_final.csv")
igdb_steam_games_final = pd.read_csv("../../data/igdb_steam_games_final.csv")

print(df_parsed.columns)
print(df_games_final.columns)
print(df_popular_final.columns)
print(df_web_final.columns)
print(igdb_steam_games_final.columns)

Index(['steam_id', 'game_name', 'game_description_snippet', 'game_price',
       'found_game_price', 'all_language_reviews_type',
       'all_language_reviews_count', 'has_russian_reviews',
       'all_russian_reviews_type', 'all_russian_reviews_count',
       'steam_app_url', 'support_ru_region'],
      dtype='str')
Index(['id', 'first_release_date', 'genres', 'name', 'rating', 'rating_count',
       'total_rating', 'total_rating_count'],
      dtype='str')
Index(['id', 'game_id', 'value', 'external_popularity_source'], dtype='str')
Index(['id', 'game', 'type'], dtype='str')
Index(['uid', 'name', 'id', 'game'], dtype='str')


Для начала сделаем merge нашей основной таблицы df_parsed с igdb_steam_games_final. Поскольку в будущем, нам нужно будет делать merge с таблицами для обогащения, полученных с api запросов (df_games_finalб df_popular_final, df_web_final). Нам нужно добавить id игр с igbd в df_parsed, чтобы в дальнейшем осуществить merge с остальными. 

Проверим пустые значения, уникальные значения и дубли перед мерджем

In [146]:
print(df_parsed["steam_id"].isna().sum())
print(df_parsed["steam_id"].nunique())
print(df_parsed.duplicated(subset=["steam_id"]).sum())

print(igdb_steam_games_final["uid"].isna().sum())
print(igdb_steam_games_final["uid"].nunique())
print(igdb_steam_games_final.duplicated(subset=["uid"]).sum())

0
26853
0
0
25701
1


In [147]:
igdb_steam_games_final[igdb_steam_games_final["uid"].duplicated(keep=False)]

,uid,name,id,game
45,1438480,Saviorless,2451784.0,25338
46,1438480,Saviorless,2048921.0,25338


In [148]:
igdb_steam_games_final = igdb_steam_games_final.drop(columns = {"id"})

In [149]:
igdb_steam_games_final

,uid,name,game
0,1313940,My Holiday 2,150866
1,1902870,Vertical Kingdom,202602
2,1071940,Streamer's Life,124447
3,1706570,Chupacabras: Night Hunt,163904
4,1085150,Golf Defied,118634
...,...,...,...
25697,1547900,Active Mummy,156747
25698,2238360,You Have No Time,244854
25699,1490030,Cyber Dodge,159989
25700,1855190,Vektor Z,186672


Нашли дубликат, проверили, эта одна и та же игра с одинаковыми id на igdb и steam, разные только id, оставшийся с 
таблицы(при сохранении не прописали index = False). Его дропнули по причине ненадобности, оставшиеся данные нужны,
по ним будем осуществлять первый мердж (по uid (steam id)), и в дальнейшем по game(igdb id). 

Удалим дубликат:

In [150]:
igdb_steam_games_final = igdb_steam_games_final.drop_duplicates()
igdb_steam_games_final[igdb_steam_games_final["uid"].duplicated(keep=False)]

,uid,name,game


In [151]:
print(igdb_steam_games_final["uid"].isna().sum())
print(igdb_steam_games_final["uid"].nunique())
print(igdb_steam_games_final.duplicated(subset=["uid"]).sum())

0
25701
0


Отлично, приступаем в merge

In [152]:
df_parsed_1 =df_parsed.merge(igdb_steam_games_final, left_on="steam_id", right_on="uid",how="inner")
df_parsed_1

,steam_id,game_name,game_description_snippet,game_price,found_game_price,all_language_reviews_type,all_language_reviews_count,has_russian_reviews,all_russian_reviews_type,all_russian_reviews_count,steam_app_url,support_ru_region,uid,name,game
0,1313940,My Holiday 2,This is the second part of the long holiday of...,259 руб.,1.0,Очень положительные,106.0,0.0,NaN,NaN,https://store.steampowered.com/app/1313940/,1.0,1313940,My Holiday 2,150866
1,1902870,Vertical Kingdom,Возродите былое величие Империи в этой карточн...,710 руб.,1.0,В основном положительные,111.0,0.0,NaN,NaN,https://store.steampowered.com/app/1902870/,1.0,1902870,Vertical Kingdom,202602
2,1071940,Streamer's Life,"Игра в жанре RPG-Simulation, в которой вам пре...",NaN,0.0,В основном положительные,139.0,0.0,NaN,NaN,https://store.steampowered.com/app/1071940/,1.0,1071940,Streamer's Life,124447
3,1706570,Chupacabras: Night Hunt,Hunt the chupacabras alone or with your friend...,99 руб.,1.0,В основном положительные,124.0,0.0,NaN,NaN,https://store.steampowered.com/app/1706570/,1.0,1706570,Chupacabras: Night Hunt,163904
4,1085150,Golf Defied,"Физика и механики нового поколения, мультиплее...",Бесплатные,1.0,Смешанные,175.0,0.0,NaN,NaN,https://store.steampowered.com/app/1085150/,1.0,1085150,Golf Defied,118634
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25696,1547900,Active Mummy,"Active Mummy — динамичный 2D-платформер, в кот...",210 руб.,1.0,Положительные,20.0,0.0,NaN,NaN,https://store.steampowered.com/app/1547900/,1.0,1547900,Active Mummy,156747
25697,2238360,You Have No Time,Rev up your inner speedster in this high-octan...,165 руб.,1.0,Положительные,13.0,0.0,NaN,NaN,https://store.steampowered.com/app/2238360/,1.0,2238360,You Have No Time,244854
25698,1490030,Cyber Dodge,Cyber Dodge is a skill-based Runner and a Die ...,61 руб.,1.0,Обзоров пользователей: 3,3.0,0.0,NaN,NaN,https://store.steampowered.com/app/1490030/,1.0,1490030,Cyber Dodge,159989
25699,1855190,Vektor Z,Vektor Z brings fast and furious retro space b...,280 руб.,1.0,Обзоров пользователей: 2,2.0,0.0,NaN,NaN,https://store.steampowered.com/app/1855190/,1.0,1855190,Vektor Z,186672


Видим датасет получившийся, нужно убрать uid (дублирует steam_id), name (дублирует game_name), и для наглядности переименуем game в igdb_id

In [153]:
df_parsed_1 = df_parsed_1.drop(columns = {"name", "uid"})
df_parsed_1 = df_parsed_1.rename(columns = {"game": "igdb_id"})
df_parsed_1

,steam_id,game_name,game_description_snippet,game_price,found_game_price,all_language_reviews_type,all_language_reviews_count,has_russian_reviews,all_russian_reviews_type,all_russian_reviews_count,steam_app_url,support_ru_region,igdb_id
0,1313940,My Holiday 2,This is the second part of the long holiday of...,259 руб.,1.0,Очень положительные,106.0,0.0,NaN,NaN,https://store.steampowered.com/app/1313940/,1.0,150866
1,1902870,Vertical Kingdom,Возродите былое величие Империи в этой карточн...,710 руб.,1.0,В основном положительные,111.0,0.0,NaN,NaN,https://store.steampowered.com/app/1902870/,1.0,202602
2,1071940,Streamer's Life,"Игра в жанре RPG-Simulation, в которой вам пре...",NaN,0.0,В основном положительные,139.0,0.0,NaN,NaN,https://store.steampowered.com/app/1071940/,1.0,124447
3,1706570,Chupacabras: Night Hunt,Hunt the chupacabras alone or with your friend...,99 руб.,1.0,В основном положительные,124.0,0.0,NaN,NaN,https://store.steampowered.com/app/1706570/,1.0,163904
4,1085150,Golf Defied,"Физика и механики нового поколения, мультиплее...",Бесплатные,1.0,Смешанные,175.0,0.0,NaN,NaN,https://store.steampowered.com/app/1085150/,1.0,118634
...,...,...,...,...,...,...,...,...,...,...,...,...,...
25696,1547900,Active Mummy,"Active Mummy — динамичный 2D-платформер, в кот...",210 руб.,1.0,Положительные,20.0,0.0,NaN,NaN,https://store.steampowered.com/app/1547900/,1.0,156747
25697,2238360,You Have No Time,Rev up your inner speedster in this high-octan...,165 руб.,1.0,Положительные,13.0,0.0,NaN,NaN,https://store.steampowered.com/app/2238360/,1.0,244854
25698,1490030,Cyber Dodge,Cyber Dodge is a skill-based Runner and a Die ...,61 руб.,1.0,Обзоров пользователей: 3,3.0,0.0,NaN,NaN,https://store.steampowered.com/app/1490030/,1.0,159989
25699,1855190,Vektor Z,Vektor Z brings fast and furious retro space b...,280 руб.,1.0,Обзоров пользователей: 2,2.0,0.0,NaN,NaN,https://store.steampowered.com/app/1855190/,1.0,186672


Перейдем к следующему датасету (df_web_final) и присоединим к общему колонку type, которая в свою очередь показывае, на какой платформе или каком сайте у игры есть страница.

In [154]:
print(df_web_final.isna().sum())
print(df_web_final.duplicated().sum()) #видим 10 дублей, посмотрим их

id      0
game    0
type    0
dtype: int64
12


In [155]:
df_web_final[df_web_final.duplicated(keep=False)] 


,id,game,type
3064,390579,75067,Twitch
3084,92133,75067,Official Website
3237,979505,75067,YouTube
3238,979504,75067,Twitter
4076,464053,159361,Twitch
10061,800116,33027,Nintendo
10190,960612,33027,Twitter
10532,170916,138871,GOG
10533,159645,138871,Community Wiki
10534,155929,138871,Steam


In [156]:
df_web_final = df_web_final.drop_duplicates()
print(df_web_final[df_web_final.duplicated(keep=False)])
df_web_final.info()

Empty DataFrame
Columns: [id, game, type]
Index: []
<class 'pandas.DataFrame'>
Index: 25988 entries, 0 to 25999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      25988 non-null  int64
 1   game    25988 non-null  int64
 2   type    25988 non-null  str  
dtypes: int64(2), str(1)
memory usage: 812.1 KB


Есть проблема, в датасете оказалось так, что у одной игры несколько таких площадок и айди дублируются, но с разными площадками, и при попытке смерджить, добавится много ненужных строк, поэтому нужно сначала аггрегировать, датасет уменьшится, и получатся много пропусков в мердженном датасете. Тем не менее, на данном этапе, мы с этим уже ничего не сделаем. Соединим по айди все площадки, и запишем их через запятую, далее можно мердж

In [157]:
df_web = (df_web_final[["game", "type"]].drop_duplicates(["game","type"]).groupby("game", as_index=False)["type"].agg(", ".join))
df_web

,game,type
0,111,Wikipedia
1,527,"YouTube, Facebook"
2,671,Official Website
3,857,Wikipedia
4,858,"Wikipedia, Community Wiki, GOG"
...,...,...
12567,393467,Steam
12568,393587,"Steam, Twitter"
12569,395919,Steam
12570,397479,Steam


In [158]:
df_parsed_2 =df_parsed_1.merge(df_web, left_on="igdb_id", right_on="game",how="left")
df_parsed_2 = df_parsed_2.drop(columns= {"game"})
df_parsed_2

,steam_id,game_name,game_description_snippet,game_price,found_game_price,all_language_reviews_type,all_language_reviews_count,has_russian_reviews,all_russian_reviews_type,all_russian_reviews_count,steam_app_url,support_ru_region,igdb_id,type
0,1313940,My Holiday 2,This is the second part of the long holiday of...,259 руб.,1.0,Очень положительные,106.0,0.0,NaN,NaN,https://store.steampowered.com/app/1313940/,1.0,150866,Official Website
1,1902870,Vertical Kingdom,Возродите былое величие Империи в этой карточн...,710 руб.,1.0,В основном положительные,111.0,0.0,NaN,NaN,https://store.steampowered.com/app/1902870/,1.0,202602,"GOG, Nintendo"
2,1071940,Streamer's Life,"Игра в жанре RPG-Simulation, в которой вам пре...",NaN,0.0,В основном положительные,139.0,0.0,NaN,NaN,https://store.steampowered.com/app/1071940/,1.0,124447,Steam
3,1706570,Chupacabras: Night Hunt,Hunt the chupacabras alone or with your friend...,99 руб.,1.0,В основном положительные,124.0,0.0,NaN,NaN,https://store.steampowered.com/app/1706570/,1.0,163904,NaN
4,1085150,Golf Defied,"Физика и механики нового поколения, мультиплее...",Бесплатные,1.0,Смешанные,175.0,0.0,NaN,NaN,https://store.steampowered.com/app/1085150/,1.0,118634,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25696,1547900,Active Mummy,"Active Mummy — динамичный 2D-платформер, в кот...",210 руб.,1.0,Положительные,20.0,0.0,NaN,NaN,https://store.steampowered.com/app/1547900/,1.0,156747,Steam
25697,2238360,You Have No Time,Rev up your inner speedster in this high-octan...,165 руб.,1.0,Положительные,13.0,0.0,NaN,NaN,https://store.steampowered.com/app/2238360/,1.0,244854,"Twitter, Discord, YouTube, Official Website"
25698,1490030,Cyber Dodge,Cyber Dodge is a skill-based Runner and a Die ...,61 руб.,1.0,Обзоров пользователей: 3,3.0,0.0,NaN,NaN,https://store.steampowered.com/app/1490030/,1.0,159989,Steam
25699,1855190,Vektor Z,Vektor Z brings fast and furious retro space b...,280 руб.,1.0,Обзоров пользователей: 2,2.0,0.0,NaN,NaN,https://store.steampowered.com/app/1855190/,1.0,186672,"Discord, Official Website, Steam"


In [159]:
df_parsed_2.isna().sum()

steam_id                          0
game_name                         0
game_description_snippet        522
game_price                     5896
found_game_price                954
all_language_reviews_type      1320
all_language_reviews_count     1320
has_russian_reviews             954
all_russian_reviews_type      24167
all_russian_reviews_count     24167
steam_app_url                   954
support_ru_region               954
igdb_id                           0
type                          13116
dtype: int64

Перейдем к следующему датасету

In [160]:
df_games_final.info()
df_games_final

<class 'pandas.DataFrame'>
RangeIndex: 25698 entries, 0 to 25697
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  25698 non-null  int64  
 1   first_release_date  25364 non-null  float64
 2   genres              25531 non-null  str    
 3   name                25698 non-null  str    
 4   rating              10120 non-null  float64
 5   rating_count        10120 non-null  float64
 6   total_rating        10554 non-null  float64
 7   total_rating_count  10554 non-null  float64
dtypes: float64(5), int64(1), str(2)
memory usage: 1.6 MB


,id,first_release_date,genres,name,rating,rating_count,total_rating,total_rating_count
0,9012,1.386115e+09,"Puzzle, Indie",Robot Rescue Revolution,NaN,NaN,NaN,NaN
1,9865,1.399334e+09,"Fighting, Sport, Indie, Arcade",Sportsfriends,62.152051,21.0,71.790311,28.0
2,10336,1.640822e+09,"Puzzle, Strategy, Indie, Card & Board Game",Frogs vs. Storks,NaN,NaN,NaN,NaN
3,11644,1.439165e+09,"Simulator, Adventure, Indie",One Final Breath,NaN,NaN,NaN,NaN
4,13359,1.445299e+09,"Adventure, Indie",Pulse,40.000000,1.0,50.833333,4.0
...,...,...,...,...,...,...,...,...
25693,323460,1.736381e+09,"Role-playing (RPG), Indie",The Rangers in the South,NaN,NaN,NaN,NaN
25694,327806,1.734048e+09,"Puzzle, Simulator, Indie",Night at the Office,NaN,NaN,NaN,NaN
25695,356141,1.741392e+09,"Puzzle, Adventure, Indie",Exit The Abyss,NaN,NaN,NaN,NaN
25696,367683,1.568160e+09,Indie,Ain Dodo,NaN,NaN,NaN,NaN


Зная, из документации IGDB, что тип данных у first_release_year - это Unix Time Stamp, модифицируем дату 

In [161]:
df_games_final["first_release_date"] = pd.to_datetime(df_games_final["first_release_date"], unit= "s").dt.date
df_games_final #норм, мерджим пока что

,id,first_release_date,genres,name,rating,rating_count,total_rating,total_rating_count
0,9012,2013-12-04,"Puzzle, Indie",Robot Rescue Revolution,NaN,NaN,NaN,NaN
1,9865,2014-05-06,"Fighting, Sport, Indie, Arcade",Sportsfriends,62.152051,21.0,71.790311,28.0
2,10336,2021-12-30,"Puzzle, Strategy, Indie, Card & Board Game",Frogs vs. Storks,NaN,NaN,NaN,NaN
3,11644,2015-08-10,"Simulator, Adventure, Indie",One Final Breath,NaN,NaN,NaN,NaN
4,13359,2015-10-20,"Adventure, Indie",Pulse,40.000000,1.0,50.833333,4.0
...,...,...,...,...,...,...,...,...
25693,323460,2025-01-09,"Role-playing (RPG), Indie",The Rangers in the South,NaN,NaN,NaN,NaN
25694,327806,2024-12-13,"Puzzle, Simulator, Indie",Night at the Office,NaN,NaN,NaN,NaN
25695,356141,2025-03-08,"Puzzle, Adventure, Indie",Exit The Abyss,NaN,NaN,NaN,NaN
25696,367683,2019-09-11,Indie,Ain Dodo,NaN,NaN,NaN,NaN


In [162]:
df_parsed_3 =df_parsed_2.merge(df_games_final, left_on="igdb_id", right_on="id",how="left")
df_parsed_3 = df_parsed_3.drop(columns= {"id", "name"})
df_parsed_3

,steam_id,game_name,game_description_snippet,game_price,found_game_price,all_language_reviews_type,all_language_reviews_count,has_russian_reviews,all_russian_reviews_type,all_russian_reviews_count,steam_app_url,support_ru_region,igdb_id,type,first_release_date,genres,rating,rating_count,total_rating,total_rating_count
0,1313940,My Holiday 2,This is the second part of the long holiday of...,259 руб.,1.0,Очень положительные,106.0,0.0,NaN,NaN,https://store.steampowered.com/app/1313940/,1.0,150866,Official Website,2020-10-05,Indie,NaN,NaN,NaN,NaN
1,1902870,Vertical Kingdom,Возродите былое величие Империи в этой карточн...,710 руб.,1.0,В основном положительные,111.0,0.0,NaN,NaN,https://store.steampowered.com/app/1902870/,1.0,202602,"GOG, Nintendo",2024-04-15,"Puzzle, Strategy, Indie, Card & Board Game",NaN,NaN,NaN,NaN
2,1071940,Streamer's Life,"Игра в жанре RPG-Simulation, в которой вам пре...",NaN,0.0,В основном положительные,139.0,0.0,NaN,NaN,https://store.steampowered.com/app/1071940/,1.0,124447,Steam,2019-07-31,"Role-playing (RPG), Simulator, Indie",NaN,NaN,NaN,NaN
3,1706570,Chupacabras: Night Hunt,Hunt the chupacabras alone or with your friend...,99 руб.,1.0,В основном положительные,124.0,0.0,NaN,NaN,https://store.steampowered.com/app/1706570/,1.0,163904,NaN,2021-08-20,"Shooter, Indie",NaN,NaN,NaN,NaN
4,1085150,Golf Defied,"Физика и механики нового поколения, мультиплее...",Бесплатные,1.0,Смешанные,175.0,0.0,NaN,NaN,https://store.steampowered.com/app/1085150/,1.0,118634,NaN,2019-03-11,"Simulator, Sport, Indie",NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25734,1547900,Active Mummy,"Active Mummy — динамичный 2D-платформер, в кот...",210 руб.,1.0,Положительные,20.0,0.0,NaN,NaN,https://store.steampowered.com/app/1547900/,1.0,156747,Steam,2021-02-28,"Sport, Adventure, Indie",NaN,NaN,NaN,NaN
25735,2238360,You Have No Time,Rev up your inner speedster in this high-octan...,165 руб.,1.0,Положительные,13.0,0.0,NaN,NaN,https://store.steampowered.com/app/2238360/,1.0,244854,"Twitter, Discord, YouTube, Official Website",2023-11-09,"Puzzle, Indie",NaN,NaN,NaN,NaN
25736,1490030,Cyber Dodge,Cyber Dodge is a skill-based Runner and a Die ...,61 руб.,1.0,Обзоров пользователей: 3,3.0,0.0,NaN,NaN,https://store.steampowered.com/app/1490030/,1.0,159989,Steam,2021-02-14,Indie,NaN,NaN,NaN,NaN
25737,1855190,Vektor Z,Vektor Z brings fast and furious retro space b...,280 руб.,1.0,Обзоров пользователей: 2,2.0,0.0,NaN,NaN,https://store.steampowered.com/app/1855190/,1.0,186672,"Discord, Official Website, Steam",2022-01-14,"Shooter, Indie, Arcade",NaN,NaN,NaN,NaN


Теперь датасет df_popular

In [163]:
df_popular_final['id'].duplicated().sum()

np.int64(10)

In [164]:
df_popular_final[df_popular_final.duplicated(keep=False)] 

,id,game_id,value,external_popularity_source
478,4088737,19243,0.000002,IGDB
3204,7974,75067,0.000009,IGDB
3205,122614,75067,0.000007,IGDB
3206,3518883,75067,0.000019,IGDB
6118,4088737,19243,0.000002,IGDB
7352,4353838,101240,0.000002,IGDB
7353,4790619,101240,0.000004,IGDB
17064,7974,75067,0.000009,IGDB
17065,122614,75067,0.000007,IGDB
17066,3518883,75067,0.000019,IGDB


In [165]:
df_popular_final[df_popular_final[['game_id']].duplicated(keep=False)] 

,id,game_id,value,external_popularity_source
1,3372,9865,2.828108e-05,IGDB
2,98958,9865,1.646263e-05,IGDB
3,4845023,9865,1.553492e-06,IGDB
4,4751475,9865,4.206028e-06,IGDB
7,3807816,13739,7.839350e-07,IGDB
...,...,...,...,...
23724,4576780,225534,1.261808e-05,IGDB
23725,4793010,132925,4.206028e-06,IGDB
23727,4827841,22441,4.206028e-06,IGDB
23728,4833365,182028,4.206028e-06,IGDB


In [166]:
df_popular_final[df_popular_final[['game_id', 'external_popularity_source']].duplicated(keep=False)] 

,id,game_id,value,external_popularity_source
1,3372,9865,2.828108e-05,IGDB
2,98958,9865,1.646263e-05,IGDB
3,4845023,9865,1.553492e-06,IGDB
4,4751475,9865,4.206028e-06,IGDB
7,3807816,13739,7.839350e-07,IGDB
...,...,...,...,...
23724,4576780,225534,1.261808e-05,IGDB
23725,4793010,132925,4.206028e-06,IGDB
23727,4827841,22441,4.206028e-06,IGDB
23728,4833365,182028,4.206028e-06,IGDB


In [167]:
df_popular_final['game_id'].nunique() 

11606

Заметим, что хотя и строк довольно много, фактически это все данные для 4531 игры, так что сгруппируем датасет по играм, а также источнику, т.к. для одной игры может быть описана информация с 1 и более источника

In [168]:
df_popular_sum_groupby_source = df_popular_final.groupby(['game_id', 'external_popularity_source'])['value'].sum().reset_index().sort_values(by=['game_id', 'value'], ascending=[True, False])
df_popular_sum_groupby_source

,game_id,external_popularity_source,value
0,111,IGDB,0.001009
1,111,Steam,0.000125
2,111,Twitch,0.000007
3,232,IGDB,0.000681
4,232,Steam,0.000135
...,...,...,...
13159,389548,IGDB,0.000004
13160,390164,IGDB,0.000002
13161,390166,IGDB,0.000004
13162,394425,IGDB,0.000004


In [169]:
df_popular_sum_groupby_source['external_popularity_source'].value_counts()

external_popularity_source
IGDB      11542
Twitch     1137
Steam       485
Name: count, dtype: int64

У нас всего 3 уникальных источника. Выделим в отдельные столбцы все уникальные значения источников для каждой представленной там игры. И уже эту таблицу будем мерджить с общим датасетом.

In [170]:
df_popular_pivot = df_popular_sum_groupby_source.pivot(index='game_id', columns='external_popularity_source', values='value').reset_index()

df_popular_pivot = df_popular_pivot.rename(columns={'IGDB': 'popularity_igdb', 'Steam': 'popularity_steam', 'Twitch': 'popularity_twitch'})

In [171]:
df_popular_pivot

external_popularity_source,game_id,popularity_igdb,popularity_steam,popularity_twitch
0,111,0.001009,0.000125,0.000007
1,232,0.000681,0.000135,0.000003
2,527,0.000345,0.000050,0.000004
3,532,0.000150,NaN,NaN
4,576,0.000253,0.000234,0.000023
...,...,...,...,...
11601,389548,0.000004,NaN,NaN
11602,390164,0.000002,NaN,NaN
11603,390166,0.000004,NaN,NaN
11604,394425,0.000004,NaN,NaN


Смерджим с общим датасетом

In [172]:
df_final = df_parsed_3.merge(df_popular_pivot, left_on='igdb_id', right_on='game_id', how='left')
df_final = df_final.drop(columns={'game_id'})

Проверим финально на дубликаты

In [173]:
df_final[df_final.duplicated(keep=False)]

,steam_id,game_name,game_description_snippet,game_price,found_game_price,all_language_reviews_type,all_language_reviews_count,has_russian_reviews,all_russian_reviews_type,all_russian_reviews_count,...,type,first_release_date,genres,rating,rating_count,total_rating,total_rating_count,popularity_igdb,popularity_steam,popularity_twitch
586,1572790,Gustavo : Kingdom Rebirth,"The Gustavo Empire, an empire ruled by women, ...",154 руб.,1.0,Обзоров пользователей: 7,7.0,0.0,NaN,NaN,...,Official Website,2021-03-25,"Role-playing (RPG), Adventure, Indie",NaN,NaN,NaN,NaN,NaN,NaN,NaN
587,1572790,Gustavo : Kingdom Rebirth,"The Gustavo Empire, an empire ruled by women, ...",154 руб.,1.0,Обзоров пользователей: 7,7.0,0.0,NaN,NaN,...,Official Website,2021-03-25,"Role-playing (RPG), Adventure, Indie",NaN,NaN,NaN,NaN,NaN,NaN,NaN
701,1289060,No One But You,A Visual Novel/Dating Sim about Hideaki who ha...,259 руб.,1.0,Обзоров пользователей: 4,4.0,0.0,NaN,NaN,...,"Steam, Official Website",2016-01-19,"Simulator, Adventure, Indie, Visual Novel",80.000000,1.0,80.000000,1.0,0.000003,NaN,NaN
702,1289060,No One But You,A Visual Novel/Dating Sim about Hideaki who ha...,259 руб.,1.0,Обзоров пользователей: 4,4.0,0.0,NaN,NaN,...,"Steam, Official Website",2016-01-19,"Simulator, Adventure, Indie, Visual Novel",80.000000,1.0,80.000000,1.0,0.000003,NaN,NaN
3460,1209860,BDSM: Big Drunk Satanic Massacre Demo,"BDSM - первый сатирический экшн-РПГ, в котором...",Бесплатно,1.0,Очень положительные,195.0,0.0,NaN,NaN,...,"Twitch, Official Website, YouTube, Twitter, Steam",2019-10-10,"Shooter, Role-playing (RPG), Adventure, Indie",67.292505,6.0,63.646252,7.0,0.000070,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23807,1157940,Gnomes & Goblins,Embark on a dream-like journey in this fantasy...,435 руб.,1.0,В основном положительные,102.0,0.0,NaN,NaN,...,YouTube,2020-09-23,"Simulator, Adventure, Indie",NaN,NaN,NaN,NaN,0.000007,NaN,NaN
24061,895980,《古斯塔奥：帝國重生》Gustavo Kingdom Rebirth,The story is mainly about the Gustavo empire. ...,NaN,0.0,Смешанные,16.0,0.0,NaN,NaN,...,Official Website,2021-03-25,"Role-playing (RPG), Adventure, Indie",NaN,NaN,NaN,NaN,NaN,NaN,NaN
24062,895980,《古斯塔奥：帝國重生》Gustavo Kingdom Rebirth,The story is mainly about the Gustavo empire. ...,NaN,0.0,Смешанные,16.0,0.0,NaN,NaN,...,Official Website,2021-03-25,"Role-playing (RPG), Adventure, Indie",NaN,NaN,NaN,NaN,NaN,NaN,NaN
24683,2525,Gumboy - Crazy Adventures™,Gumboy обладает веселым и новаторским игровым ...,200 руб.,1.0,Смешанные,108.0,0.0,NaN,NaN,...,NaN,2006-12-19,"Puzzle, Indie",50.000000,4.0,50.000000,4.0,0.000022,NaN,NaN


Нашли полные дубликаты, очистим их. Возможно, на прошлых стадиях мерджа что-то было упущено.

In [174]:
df_final = df_final.drop_duplicates()

In [175]:
df_final[df_final.duplicated(keep=False)]

,steam_id,game_name,game_description_snippet,game_price,found_game_price,all_language_reviews_type,all_language_reviews_count,has_russian_reviews,all_russian_reviews_type,all_russian_reviews_count,...,type,first_release_date,genres,rating,rating_count,total_rating,total_rating_count,popularity_igdb,popularity_steam,popularity_twitch


In [176]:
df_final[df_final['steam_id'].duplicated(keep=False)]

,steam_id,game_name,game_description_snippet,game_price,found_game_price,all_language_reviews_type,all_language_reviews_count,has_russian_reviews,all_russian_reviews_type,all_russian_reviews_count,...,type,first_release_date,genres,rating,rating_count,total_rating,total_rating_count,popularity_igdb,popularity_steam,popularity_twitch


Отлично, ни полных, ни по steam_id дубликатов не осталось. Можно выгружать датасет. 

In [177]:
df_final

,steam_id,game_name,game_description_snippet,game_price,found_game_price,all_language_reviews_type,all_language_reviews_count,has_russian_reviews,all_russian_reviews_type,all_russian_reviews_count,...,type,first_release_date,genres,rating,rating_count,total_rating,total_rating_count,popularity_igdb,popularity_steam,popularity_twitch
0,1313940,My Holiday 2,This is the second part of the long holiday of...,259 руб.,1.0,Очень положительные,106.0,0.0,NaN,NaN,...,Official Website,2020-10-05,Indie,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1902870,Vertical Kingdom,Возродите былое величие Империи в этой карточн...,710 руб.,1.0,В основном положительные,111.0,0.0,NaN,NaN,...,"GOG, Nintendo",2024-04-15,"Puzzle, Strategy, Indie, Card & Board Game",NaN,NaN,NaN,NaN,4.674854e-06,NaN,NaN
2,1071940,Streamer's Life,"Игра в жанре RPG-Simulation, в которой вам пре...",NaN,0.0,В основном положительные,139.0,0.0,NaN,NaN,...,Steam,2019-07-31,"Role-playing (RPG), Simulator, Indie",NaN,NaN,NaN,NaN,NaN,NaN,0.0
3,1706570,Chupacabras: Night Hunt,Hunt the chupacabras alone or with your friend...,99 руб.,1.0,В основном положительные,124.0,0.0,NaN,NaN,...,NaN,2021-08-20,"Shooter, Indie",NaN,NaN,NaN,NaN,4.206028e-06,NaN,NaN
4,1085150,Golf Defied,"Физика и механики нового поколения, мультиплее...",Бесплатные,1.0,Смешанные,175.0,0.0,NaN,NaN,...,NaN,2019-03-11,"Simulator, Sport, Indie",NaN,NaN,NaN,NaN,7.839350e-07,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25734,1547900,Active Mummy,"Active Mummy — динамичный 2D-платформер, в кот...",210 руб.,1.0,Положительные,20.0,0.0,NaN,NaN,...,Steam,2021-02-28,"Sport, Adventure, Indie",NaN,NaN,NaN,NaN,NaN,NaN,NaN
25735,2238360,You Have No Time,Rev up your inner speedster in this high-octan...,165 руб.,1.0,Положительные,13.0,0.0,NaN,NaN,...,"Twitter, Discord, YouTube, Official Website",2023-11-09,"Puzzle, Indie",NaN,NaN,NaN,NaN,NaN,NaN,NaN
25736,1490030,Cyber Dodge,Cyber Dodge is a skill-based Runner and a Die ...,61 руб.,1.0,Обзоров пользователей: 3,3.0,0.0,NaN,NaN,...,Steam,2021-02-14,Indie,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25737,1855190,Vektor Z,Vektor Z brings fast and furious retro space b...,280 руб.,1.0,Обзоров пользователей: 2,2.0,0.0,NaN,NaN,...,"Discord, Official Website, Steam",2022-01-14,"Shooter, Indie, Arcade",NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [178]:
df_final.to_csv('../../data/df_final.csv', index=False)